In [1]:
### 0. IMPORTAÇÃO DE BIBLIOTECAS ###
import pandas as pd
import numpy as np
import unicodedata
import re
import sys

print(f'Executável do Python: {sys.executable}')

Executável do Python: C:\Users\luish\anaconda3\envs\bs_fornecedor\python.exe


In [2]:
### 1. CARGA DOS DADOS BRUTOS ###
df = pd.read_excel(r"C:\Users\luish\Downloads\Desenvolvimento TCC\TCC\01 - Dados Brutos\Secundarios\Base de dados\base_fornecedor_v1.xlsx")
print(f'Dataset Principal carregado: {df.shape[0]} linhas e {df.shape[1]} colunas.\n')

Dataset Principal carregado: 2011 linhas e 39 colunas.



In [3]:
### 2. EXPLORAÇÃO E DIAGNÓSTICO INICIAL ###
print("="*50)
print("--- DIAGNÓSTICO INICIAL DOS DADOS ---")
print("="*50)

print("\n[INFO] Estrutura do DataFrame:")
df.info()

print("\n[INFO] Primeiras 3 linhas:")
print(df.head(3))

print("\n[INFO] Percentual de Valores Nulos por Coluna:")
nulos_pct = (df.isnull().sum() / len(df)) * 100
print(nulos_pct[nulos_pct > 0].round(2).to_string() if not nulos_pct[nulos_pct > 0].empty else "Nenhum valor nulo encontrado.")

print("\n[INFO] Total de Linhas Duplicadas no Dataset Bruto:", df.duplicated().sum())
print("="*50 + "\n")

--- DIAGNÓSTICO INICIAL DOS DADOS ---

[INFO] Estrutura do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 2011 entries, 0 to 2010
Data columns (total 39 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   CNPJ                                  2011 non-null   int64  
 1   Raiz CNPJ                             2011 non-null   int64  
 2   Numero da Filial                      2011 non-null   int64  
 3   Razao Social                          2011 non-null   str    
 4   Matriz Filial                         2011 non-null   str    
 5   Codigo Natureza Juridica              2011 non-null   str    
 6   Descricao Natureza Juridica           2011 non-null   str    
 7   Nome Fantasia                         863 non-null    str    
 8   Situacao Cadastral                    2011 non-null   str    
 9   Data Situacao Cadastral               2007 non-null   object 
 10  Motivo Situacao Cadastral

In [4]:
### 3. PADRONIZAÇÃO E NORMALIZAÇÃO DE NOMES DAS COLUNAS ###
def normalizar_nome_coluna(col):
    # Remove acentos e caracteres especiais
    nfkd = unicodedata.normalize('NFKD', col)
    texto_sem_acento = u"".join([c for c in nfkd if not unicodedata.combining(c)])
    # Substitui caracteres não alfanuméricos por underline e converte para minúsculas
    texto_limpo = re.sub(r'[^a-zA-Z0-9]+', '_', texto_sem_acento).strip('_').lower()
    return texto_limpo

# Aplicar normalização nos nomes das colunas (Ex: 'Razão Social' -> 'razao_social')
df.columns = [normalizar_nome_coluna(col) for col in df.columns]
print("[OK] Nomes de colunas padronizados para snake_case e sem acentos.")

[OK] Nomes de colunas padronizados para snake_case e sem acentos.


In [5]:
### 4. AJUSTE DE TIPAGEM (DATA TYPES) ###
df_limpo = df.copy()

if 'cnpj' in df_limpo.columns:
    df_limpo['cnpj'] = df_limpo['cnpj'].astype(str).str.zfill(14)
if 'raiz_cnpj' in df_limpo.columns:
    df_limpo['raiz_cnpj'] = df_limpo['raiz_cnpj'].astype(str).str.zfill(8)
if 'cep' in df_limpo.columns:
    df_limpo['cep'] = df_limpo['cep'].astype(str).str.zfill(8)

if 'data_de_abertura' in df_limpo.columns:
    df_limpo['data_de_abertura'] = pd.to_datetime(df_limpo['data_de_abertura'], errors='coerce')
if 'data_situacao_cadastral' in df_limpo.columns:
    df_limpo['data_situacao_cadastral'] = pd.to_datetime(df_limpo['data_situacao_cadastral'], errors='coerce')

print("[OK] Tipagem ajustada (CNPJ/CEP formatados com zeros à esquerda e datas convertidas).")

[OK] Tipagem ajustada (CNPJ/CEP formatados com zeros à esquerda e datas convertidas).


In [6]:
### 5. SANEAMENTO E FILTRAGEM DE EMPRESAS ATIVAS ###
# Filtrar apenas Situação Cadastral ATIVA
if 'situacao_cadastral' in df_limpo.columns:
    df_limpo['situacao_cadastral'] = df_limpo['situacao_cadastral'].astype(str).str.strip().str.upper()
    df_limpo = df_limpo[df_limpo['situacao_cadastral'] == 'ATIVA']
    print(f"[OK] Linhas ativas mantidas: {len(df_limpo)}")

# Remover Duplicações pelo CNPJ
if 'cnpj' in df_limpo.columns:
    df_limpo = df_limpo.drop_duplicates(subset=['cnpj'], keep='first')
    print(f"[OK] Linhas após remover CNPJs duplicados: {len(df_limpo)}")

[OK] Linhas ativas mantidas: 2011
[OK] Linhas após remover CNPJs duplicados: 2011


In [7]:
### 6. REMOÇÃO DE COLUNAS IRRELEVANTES (PÓS-DIAGNÓSTICO E PÓS-FILTRAGEM) ###
# Os nomes das colunas a remover foram atualizados para o padrão normalizado
colunas_para_remover = [
    'situacao_especial',
    'motivo_situacao_cadastral',
    'codigo_ibge_do_municipio',
    'data_da_consulta',
    'socios',
    'ente_federativo_responsavel',
    'optante_mei',
    'optante_simples',
    'situacao_cadastral'  # Removida com segurança após o filtro de 'ATIVA'
]

df_limpo = df_limpo.drop(columns=colunas_para_remover, errors='ignore')
print(f"[OK] Colunas irrelevantes removidas. Restantes: {df_limpo.shape[1]} colunas.")

[OK] Colunas irrelevantes removidas. Restantes: 30 colunas.


In [8]:
### 7. TRATAMENTO DE VALORES AUSENTES (MISSING DATA) ###
# Subtitui 'nome_fantasia' ausente ou 'NÃO INFORMADO' pelo valor da 'razao_social'
if 'nome_fantasia' in df_limpo.columns and 'razao_social' in df_limpo.columns:
    df_limpo['nome_fantasia'] = df_limpo['nome_fantasia'].replace(['', 'NÃO INFORMADO'], np.nan)
    df_limpo['nome_fantasia'] = df_limpo['nome_fantasia'].fillna(df_limpo['razao_social'])

valores_padrao = {
    'complemento': 'NÃO INFORMADO',
    'e_mail': 'NÃO INFORMADO'
}
df_limpo = df_limpo.fillna(value=valores_padrao)
print("[OK] Nome Fantasia ajustado com Razão Social e demais nulos imputados com padrão.")

[OK] Nome Fantasia ajustado com Razão Social e demais nulos imputados com padrão.


In [9]:
### 8. TRATAMENTO AVANÇADO DE COMPLEMENTO E LOGRADOURO ###
def limpar_texto_endereco(texto):
    if pd.isna(texto) or str(texto).strip() == '':
        return ''
    
    texto = str(texto).upper().strip()
    texto = re.sub(r';', ' ', texto)
    texto = re.sub(r'(QUADRA|QD|Q|LOTE|LT|L)(\d+)', r'\1 \2', texto)
    texto = re.sub(r'\bQD\.?\b', 'QUADRA', texto)
    texto = re.sub(r'\bLT\.?\b', 'LOTE', texto)
    texto = re.sub(r'\bKM\.?\b', 'KM', texto)
    texto = re.sub(r'\b(RUA)\s+\1\b', r'\1', texto)
    texto = re.sub(r'\b(AVENIDA)\s+\1\b', r'\1', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

if 'logradouro' in df_limpo.columns:
    df_limpo['logradouro'] = df_limpo['logradouro'].apply(limpar_texto_endereco)

if 'complemento' in df_limpo.columns:
    df_limpo['complemento'] = df_limpo['complemento'].apply(limpar_texto_endereco)
    df_limpo['complemento'] = df_limpo['complemento'].replace('', 'NÃO INFORMADO')

print("[OK] Limpeza de Regex aplicada em 'logradouro' e 'complemento'.")

[OK] Limpeza de Regex aplicada em 'logradouro' e 'complemento'.


In [10]:
### 9. ESTRUTURAÇÃO DE ENDEREÇO E EXPORTAÇÃO SEPARADA POR CNAE ###
if 'municipio' not in df_limpo.columns:
    df_limpo['municipio'] = 'PARAUAPEBAS'
if 'uf' not in df_limpo.columns:
    df_limpo['uf'] = 'PA'

colunas_essenciais = [
    'cnpj', 'razao_social', 'nome_fantasia', 'porte_da_empresa', 'data_de_abertura',
    'codigo_da_atividade_principal', 'logradouro', 'numero', 'complemento', 
    'bairro', 'municipio', 'uf', 'cep'
]
colunas_existentes = [col for col in colunas_essenciais if col in df_limpo.columns]
df_final = df_limpo[colunas_existentes].copy()

def montar_endereco_qgis(row):
    comp = f" - {row['complemento']}" if row['complemento'] != 'NÃO INFORMADO' else ""
    return (
        f"{row['logradouro']}, {row['numero']}{comp} - "
        f"{row['bairro']}, {row['municipio']} - {row['uf']}, "
        f"{row['cep']}, Brasil"
    )

df_final['endereco_completo'] = df_final.apply(montar_endereco_qgis, axis=1)

# Normalizar CNAE
df_final['cnae_limpo'] = df_final['codigo_da_atividade_principal'].astype(str).str.replace(r'[-./ ]', '', regex=True)

# Listas de CNAEs
cnaes_primarios_limpos = ["4644301", "4645101"]
cnaes_secundarios_limpos = ["4771702", "4772500", "4646001"]

# Filtrar e Separar
df_primarios = df_final[df_final["cnae_limpo"].isin(cnaes_primarios_limpos)].drop(columns=['cnae_limpo'])
df_secundarios = df_final[df_final["cnae_limpo"].isin(cnaes_secundarios_limpos)].drop(columns=['cnae_limpo'])

# Exportar Resultados
df_primarios.to_excel(r"C:\Users\luish\fornecedores_primarios_limpo.xlsx", index=False)
df_secundarios.to_excel(r"C:\Users\luish\fornecedores_secundarios_limpo.xlsx", index=False)

print("\n[OK] Pipeline executado com sucesso!")
print(f"-> Fornecedores Primários encontrados: {len(df_primarios)} linhas.")
print(f"-> Fornecedores Secundários encontrados: {len(df_secundarios)} linhas.")


[OK] Pipeline executado com sucesso!
-> Fornecedores Primários encontrados: 6 linhas.
-> Fornecedores Secundários encontrados: 270 linhas.
